# EDA
- 클래스 분포 시각화
- 클래스별 Train 이미지 전수 시각화
- Test 이미지 시각화 (Rotation/Flip/손상 확인)
- 이미지 품질 분석 (Blur / Noise / 밝기)

---
## STEP 0. 환경 설정

In [ ]:
import os
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2

random.seed(42)
np.random.seed(42)

# 경로 설정
PROJECT_ROOT = Path('..')

DATA_DIR   = PROJECT_ROOT / 'data'
TRAIN_DIR  = DATA_DIR / 'train'
TEST_DIR   = DATA_DIR / 'test'
TRAIN_CSV  = DATA_DIR / 'train.csv'
META_CSV   = DATA_DIR / 'meta.csv'
SAMPLE_CSV = DATA_DIR / 'sample_submission.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'eda'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 경로 확인
for name, p in [('TRAIN_DIR', TRAIN_DIR), ('TEST_DIR', TEST_DIR),
                ('TRAIN_CSV', TRAIN_CSV), ('META_CSV', META_CSV)]:
    icon = '✅' if p.exists() else '❌'
    print(f'  {icon} {name}: {p.resolve()}')

In [ ]:
# 한글 폰트
import matplotlib.font_manager as fm

font_path = os.path.join(os.path.expanduser("~"), ".fonts", "NanumGothic-Regular.ttf")
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

---
## STEP 1. 데이터 로드 & 구조 파악

In [ ]:
df_train  = pd.read_csv(TRAIN_CSV)
df_meta   = pd.read_csv(META_CSV)
df_sample = pd.read_csv(SAMPLE_CSV)

# 클래스 이름 매핑
target_to_name = dict(zip(df_meta['target'], df_meta['class_name']))
df_train['class_name'] = df_train['target'].map(target_to_name)

print('클래스 매핑 (17개):')
print('-' * 55)
for _, row in df_meta.iterrows():
    print(f"  {row['target']:2d} → {row['class_name']}")

print(f"\nTrain: {len(df_train)}장 | Test: {len(df_sample)}장")

---
## STEP 2. 클래스 분포 시각화

In [ ]:
df_train['class_name'].value_counts().sort_index()

In [ ]:
class_counts = df_train['target'].value_counts().sort_index()
colors17 = plt.cm.tab20(np.linspace(0, 1, 17))

fig, ax = plt.subplots(figsize=(16, 5))
bars = ax.bar(range(17), class_counts.values, color=colors17, edgecolor='white')
ax.set_xticks(range(17))
ax.set_xticklabels([f'{i}\n{target_to_name[i][:18]}' for i in range(17)],
                    rotation=45, ha='right', fontsize=8)
ax.set_ylabel('이미지 수')
ax.set_title(f'클래스별 이미지 수 (총 {len(df_train)}장)', fontsize=13)

for bar, cnt in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(cnt), ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## STEP 3. 클래스별 Train 이미지 전수 시각화

In [ ]:
COLS = 10  # 한 줄에 보여줄 이미지 수 (모니터 크기에 맞게 조절)

for cls_idx in range(17):
    cls_df = df_train[df_train['target'] == cls_idx].reset_index(drop=True)
    n_imgs = len(cls_df)
    n_rows = (n_imgs + COLS - 1) // COLS

    fig, axes = plt.subplots(n_rows, COLS, figsize=(COLS * 2, n_rows * 2))
    fig.suptitle(
        f'[Class {cls_idx}] {target_to_name[cls_idx]}  ({n_imgs}장)',
        fontsize=14, fontweight='bold', y=1.02
    )

    # axes를 항상 2D 배열로
    if n_rows == 1:
        axes = np.array([axes])

    for i in range(n_rows * COLS):
        r, c = divmod(i, COLS)
        ax = axes[r, c]
        ax.axis('off')

        if i < n_imgs:
            img_path = TRAIN_DIR / cls_df.loc[i, 'ID']
            if img_path.exists():
                img = Image.open(img_path)
                ax.imshow(img)
                ax.set_title(f'{img.size[0]}x{img.size[1]}', fontsize=5, pad=1)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'02_train_cls{cls_idx:02d}.png',
                dpi=100, bbox_inches='tight')
    plt.show()
    print()

---
## STEP 4. Test 이미지 시각화 (Rotation/Flip/손상 확인)

In [ ]:
test_files = sorted(df_sample['ID'].tolist())
PAGE_SIZE  = 100   # 한 페이지당 이미지 수
COLS_TEST  = 10    # 한 줄당 이미지 수

n_pages = (len(test_files) + PAGE_SIZE - 1) // PAGE_SIZE
print(f'📄 총 {len(test_files)}장 → {n_pages}페이지 ({PAGE_SIZE}장/페이지)')

In [ ]:
# # 전체 페이지 순회 출력
# for page in range(n_pages):
#     start = page * PAGE_SIZE
#     end   = min(start + PAGE_SIZE, len(test_files))
#     page_files = test_files[start:end]
#     n_imgs = len(page_files)
#     n_rows = (n_imgs + COLS_TEST - 1) // COLS_TEST

#     fig, axes = plt.subplots(n_rows, COLS_TEST,
#                              figsize=(COLS_TEST * 2, n_rows * 2))
#     fig.suptitle(
#         f'Test 이미지 [{page+1}/{n_pages}]  #{start+1}~#{end}',
#         fontsize=13, fontweight='bold', y=1.01
#     )
#     if n_rows == 1:
#         axes = np.array([axes])

#     for i in range(n_rows * COLS_TEST):
#         r, c = divmod(i, COLS_TEST)
#         ax = axes[r, c]
#         ax.axis('off')
#         if i < n_imgs:
#             img_path = TEST_DIR / page_files[i]
#             if img_path.exists():
#                 img = Image.open(img_path)
#                 ax.imshow(img)
#                 ax.set_title(f'{img.size[0]}x{img.size[1]}', fontsize=5, pad=1)

#     plt.tight_layout()
#     plt.show()
#     print()

### 💡 특정 페이지만 보기 (선택적 실행)

In [ ]:
# 📌 원하는 페이지 번호 지정 (1부터 시작)
VIEW_PAGE = 10  # ← 숫자 바꿔서

start = (VIEW_PAGE - 1) * PAGE_SIZE
end   = min(start + PAGE_SIZE, len(test_files))
page_files = test_files[start:end]
n_imgs = len(page_files)
n_rows = (n_imgs + COLS_TEST - 1) // COLS_TEST

fig, axes = plt.subplots(n_rows, COLS_TEST,
                         figsize=(COLS_TEST * 2.2, n_rows * 2.2))
fig.suptitle(
    f'Test 이미지 [{VIEW_PAGE}/{n_pages}]  #{start+1}~#{end}',
    fontsize=13, fontweight='bold', y=1.01
)
if n_rows == 1:
    axes = np.array([axes])

for i in range(n_rows * COLS_TEST):
    r, c = divmod(i, COLS_TEST)
    ax = axes[r, c]
    ax.axis('off')
    if i < n_imgs:
        img_path = TEST_DIR / page_files[i]
        if img_path.exists():
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(page_files[i][:14], fontsize=5, pad=1)

plt.tight_layout()
plt.show()

---
## STEP 5. 이미지 크기 통계 (Train vs Test)

In [ ]:
def collect_sizes(image_dir, file_list):
    records = []
    for f in file_list:
        p = image_dir / f
        if not p.exists():
            continue
        img = Image.open(p)
        w, h = img.size
        records.append({'file': f, 'width': w, 'height': h,
                        'aspect': round(w/h, 3),
                        'filesize_kb': round(p.stat().st_size/1024, 1)})
    return pd.DataFrame(records)

df_tr_size = collect_sizes(TRAIN_DIR, df_train['ID'].tolist())
df_te_size = collect_sizes(TEST_DIR,  df_sample['ID'].tolist())

for tag, df_s in [('Train', df_tr_size), ('Test', df_te_size)]:
    print(f'\n--- {tag} ({len(df_s)}장) ---')
    for col in ['width', 'height', 'aspect', 'filesize_kb']:
        s = df_s[col]
        print(f'  {col:12s} | min={s.min():<8} max={s.max():<8} '
              f'mean={s.mean():<8.1f}')

In [ ]:
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(1, 2, figsize=(10,4))

all_w = pd.concat([df_tr_size['width'], df_te_size['width']])
all_h = pd.concat([df_tr_size['height'], df_te_size['height']])
w_min_all, w_max_all = all_w.min(), all_w.max()
h_min_all, h_max_all = all_h.min(), all_h.max()

x_lim = (w_min_all - 50, w_max_all + 50)
y_lim = (h_min_all - 50, h_max_all + 50)

for ax, (name, df_s) in zip(axes, [('Train', df_tr_size), ('Test', df_te_size)]):
    ax.set_facecolor('#FFF8DC')

    # 각 이미지의 (width, height) scatter
    ax.scatter(df_s['width'], df_s['height'],
               alpha=0.5, s=15, color='green', edgecolors='darkgreen', linewidths=0.3)

    # min / max / mean
    w_min, w_max, w_mean = df_s['width'].min(), df_s['width'].max(), df_s['width'].mean()
    h_min, h_max, h_mean = df_s['height'].min(), df_s['height'].max(), df_s['height'].mean()

    # min/max 경계선
    ax.axvline(w_min, color='red', ls='--', lw=1.2, alpha=0.7)
    ax.axvline(w_max, color='red', ls='--', lw=1.2, alpha=0.7)
    ax.axhline(h_min, color='red', ls='--', lw=1.2, alpha=0.7)
    ax.axhline(h_max, color='red', ls='--', lw=1.2, alpha=0.7)

    # mean 경계선 (파란색)
    ax.axvline(w_mean, color='blue', ls='-.', lw=1.2, alpha=0.7)
    ax.axhline(h_mean, color='blue', ls='-.', lw=1.2, alpha=0.7)

    # 라벨 - min/max (빨간색)
    ax.text(w_min, y_lim[1] - 20, f'min {w_min:.0f}', color='red', fontsize=8, ha='center')
    ax.text(w_max, y_lim[1] - 20, f'max {w_max:.0f}', color='red', fontsize=8, ha='center')
    ax.text(x_lim[1] - 10, h_max, f'max {h_max:.0f}', color='red', fontsize=8, va='center', ha='right')
    ax.text(x_lim[1] - 10, h_min, f'min {h_min:.0f}', color='red', fontsize=8, va='center', ha='right')

    # 라벨 - mean (파란색)
    ax.text(w_mean, y_lim[1] - 40, f'mean {w_mean:.0f}', color='blue', fontsize=8, ha='center')
    ax.text(x_lim[1] - 10, h_mean, f'mean {h_mean:.0f}', color='blue', fontsize=8, va='center', ha='right')

    # 주황 박스 (min~max 범위)
    rect = Rectangle((w_min, h_min), w_max - w_min, h_max - h_min,
                      linewidth=1.5, edgecolor='orange', facecolor='none', ls='--')
    ax.add_patch(rect)

    # 축 설정
    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.set_xlabel('width', fontsize=12)
    ax.set_ylabel('height', fontsize=12)
    ax.set_title(f'{name} - height/width scatter plot  ({len(df_s)}장)',
                 fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_image_size_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

---
## STEP 6. 이미지 품질 분석 (Blur / Noise / 밝기 / 엣지)

In [ ]:
def analyze_quality(image_dir, file_list, sample_n=None):
    paths = [image_dir / f for f in file_list]
    if sample_n and sample_n < len(paths):
        paths = random.sample(paths, sample_n)

    records = []
    for p in paths:
        if not p.exists():
            continue
        img = cv2.imread(str(p))
        if img is None:
            continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        blur    = cv2.Laplacian(gray, cv2.CV_64F).var()
        bright  = float(np.mean(gray))
        smoothed = cv2.GaussianBlur(gray, (5, 5), 0)
        noise   = float(np.std(gray.astype(float) - smoothed.astype(float)))
        edges   = cv2.Canny(gray, 50, 150)
        edge_d  = float(np.sum(edges > 0) / edges.size)

        records.append({'file': p.name, 'blur_score': blur,
                        'brightness': bright, 'noise_level': noise,
                        'edge_density': edge_d})
    return pd.DataFrame(records)

print('⏳ Train 전체 품질 분석 중...')
df_train_q = analyze_quality(TRAIN_DIR, df_train['ID'].tolist())
print(f'✅ Train 완료: {len(df_train_q)}장')

print('⏳ Test 품질 분석 중...')
df_test_q = analyze_quality(TEST_DIR, df_sample['ID'].tolist())  # , sample_n=500
print(f'✅ Test 완료: {len(df_test_q)}장')

In [ ]:
metrics = [
    ('blur_score',   'Blur (Laplacian Var) — 낮을수록 블러'),
    ('brightness',   '밝기 (0~255)'),
    ('noise_level',  '노이즈 수준'),
    ('edge_density', '엣지 밀도 (문서 복잡도)')
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Train vs Test 이미지 품질 비교', fontsize=14, fontweight='bold')

for ax, (col, title) in zip(axes.flat, metrics):
    ax.hist(df_train_q[col], bins=35, alpha=0.55, label='Train',
            color='steelblue', density=True)
    ax.hist(df_test_q[col],  bins=35, alpha=0.55, label='Test',
            color='coral', density=True)
    ax.axvline(df_train_q[col].mean(), color='steelblue', ls='--', lw=1.2)
    ax.axvline(df_test_q[col].mean(),  color='coral',     ls='--', lw=1.2)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## STEP 7. 특이 이미지 탐지 (극단값)

In [ ]:
def show_extremes(df_q, image_dir, metric, n=6, ascending=True):
    sorted_df = df_q.sort_values(metric, ascending=ascending).head(n)
    tag = '⬇️ lowest' if ascending else '⬆️ highest'

    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    fig.suptitle(f'{tag}  {metric}  Top-{n}', fontsize=12, fontweight='bold')
    for ax, (_, row) in zip(axes.flat, sorted_df.iterrows()):
        img = cv2.imread(str(image_dir / row['file']))
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{row['file'][:14]}\n{metric}={row[metric]:.1f}", fontsize=7)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print('🔎 가장 블러한 Train 이미지')
show_extremes(df_train_q, TRAIN_DIR, 'blur_score', n=6, ascending=True)

print('🔎 가장 선명한 Train 이미지')
show_extremes(df_train_q, TRAIN_DIR, 'blur_score', n=6, ascending=False)

print('🔎 가장 노이즈 심한 Train 이미지')
show_extremes(df_train_q, TRAIN_DIR, 'noise_level', n=6, ascending=True)

print('🔎 가장 밝은 Train 이미지')
show_extremes(df_train_q, TRAIN_DIR, 'brightness', n=6, ascending=False)

print('🔎 가장 어두운 Train 이미지')
show_extremes(df_train_q, TRAIN_DIR, 'brightness', n=6, ascending=True)

In [ ]:
def show_extremes(df_q, image_dir, metric, n=6, ascending=True):
    sorted_df = df_q.sort_values(metric, ascending=ascending).head(n)
    tag = '⬇️ lowest' if ascending else '⬆️ highest'

    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    fig.suptitle(f'{tag}  {metric}  Top-{n}', fontsize=12, fontweight='bold')
    for ax, (_, row) in zip(axes.flat, sorted_df.iterrows()):
        img = cv2.imread(str(image_dir / row['file']))
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{row['file'][:14]}\n{metric}={row[metric]:.1f}", fontsize=7)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print('🔎 가장 블러한 Test 이미지')
show_extremes(df_test_q, TEST_DIR, 'blur_score', n=6, ascending=True)

print('🔎 가장 선명한 Test 이미지')
show_extremes(df_test_q, TEST_DIR, 'blur_score', n=6, ascending=False)

print('🔎 가장 노이즈 심한 Test 이미지')
show_extremes(df_test_q, TEST_DIR, 'noise_level', n=6, ascending=True)

print('🔎 가장 밝은 Test 이미지')
show_extremes(df_test_q, TEST_DIR, 'brightness', n=6, ascending=False)

print('🔎 가장 어두운 Test 이미지')
show_extremes(df_test_q, TEST_DIR, 'brightness', n=6, ascending=True)

---
## STEP 8. 클래스별 품질 분포 (boxplot)

In [ ]:
df_tq_cls = df_train_q.merge(
    df_train[['ID', 'target', 'class_name']],
    left_on='file', right_on='ID', how='left'
)

fig, axes = plt.subplots(2, 2, figsize=(20, 11))
fig.suptitle('클래스별 이미지 품질 분포 (Train)', fontsize=14, fontweight='bold')

for ax, (col, title) in zip(axes.flat, metrics):
    data = [df_tq_cls[df_tq_cls['target'] == c][col].values for c in range(17)]
    bp = ax.boxplot(data, patch_artist=True, widths=0.6)
    for patch, clr in zip(bp['boxes'], colors17):
        patch.set_facecolor(clr)
        patch.set_alpha(0.7)
    ax.set_xticklabels([str(i) for i in range(17)], fontsize=8)
    ax.set_xlabel('class')
    ax.set_title(title, fontsize=10)

plt.tight_layout()
plt.show()

---
## STEP 9. 클래스별 픽셀 강도 분포

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(22, 10))
fig.suptitle('클래스별 그레이스케일 픽셀 강도 분포', fontsize=13, fontweight='bold')

for cls_idx in range(17):
    ax = axes[cls_idx // 6, cls_idx % 6]
    cls_files = df_train[df_train['target'] == cls_idx]['ID'].tolist()
    samples = random.sample(cls_files, min(5, len(cls_files)))

    all_px = []
    for f in samples:
        img = cv2.imread(str(TRAIN_DIR / f), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            all_px.extend(img.flatten().tolist())

    if all_px:
        ax.hist(all_px, bins=50, color=colors17[cls_idx],
                edgecolor='white', alpha=0.8, density=True)
    ax.set_title(f'[{cls_idx}] {target_to_name[cls_idx][:16]}', fontsize=7)
    ax.set_xlim(0, 255)
    ax.tick_params(labelsize=5)

axes[2, 5].axis('off')
plt.tight_layout()
plt.show()